In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Instalacja niezbędnych pakietów (openpyxl jest wymagany do obsługi .xlsx)
!pip install geopy pandas openpyxl

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# 1. Wczytanie pliku Excel
nazwa_pliku_we = '/content/drive/MyDrive/Colab Notebooks/geokod2.xlsx'

try:
    df = pd.read_excel(nazwa_pliku_we)
except FileNotFoundError:
    print(f"Błąd: Nie znaleziono pliku '{nazwa_pliku_we}'. Sprawdź nazwę w panelu roboczym.")
    raise

# 2. Utworzenie zapytania adresowego
df['Pelny_Adres'] = df['Miejscowosc'].astype(str) + ', ' + df['Wojewodztwo'].astype(str) + ', ' + df['Kraj'].astype(str)

# 3. Inicjalizacja API
geolocator = Nominatim(user_agent="projekt_magisterski_geokodowanie_xlsx")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

# 4. Proces geokodowania
print("Rozpoczęto geokodowanie bazy. Proszę czekać...")
df['Lokalizacja'] = df['Pelny_Adres'].apply(geocode)

# 5. Wyodrębnienie współrzędnych WGS 84 (EPSG:4326)
df['Szerokosc_Y'] = df['Lokalizacja'].apply(lambda loc: loc.latitude if loc else None)
df['Dlugosc_X'] = df['Lokalizacja'].apply(lambda loc: loc.longitude if loc else None)

# 6. Czyszczenie i eksport do płaskiego pliku CSV dla oprogramowania GIS
df = df.drop(columns=['Lokalizacja', 'Pelny_Adres'])
plik_wyjsciowy = 'obreby_geokodowane_ostateczne.csv'
df.to_csv(plik_wyjsciowy, index=False, encoding='utf-8')

print(f"Proces zakończony. Zapisano plik: {plik_wyjsciowy}")
display(df.head())

Rozpoczęto geokodowanie bazy. Proszę czekać...
Proces zakończony. Zapisano plik: obreby_geokodowane_ostateczne.csv


,Lp,Nazwa_Obrebu,Miejscowosc,Powiat,Wojewodztwo,Kraj,Szerokosc_Y,Dlugosc_X
0,1,Kompleks Polny,Żeleźniki,milicki,dolnośląskie,Polska,51.458010,17.408690
1,2,Kompleks Zamienice Niedźwiedzice Goliszów,Chojnów,legnicki,dolnośląskie,Polska,51.272214,15.939385
2,3,Miłkowice,Miłkowice,legnicki,dolnośląskie,Polska,51.256070,16.062910
3,4,Jaśkowice Legnickie,Kunice,legnicki,dolnośląskie,Polska,51.222290,16.249893
4,5,Raków,Długołęka,wrocławski,dolnośląskie,Polska,51.173316,17.192786


In [ ]:
# Ścieżka docelowa na Dysku Google
sciezka_dysk_google = '/content/drive/MyDrive/Colab Notebooks/obreby_geokodowane_ostateczne.csv'

# Zapisanie DataFrame do pliku CSV na Dysku
df.to_csv(sciezka_dysk_google, index=False, encoding='utf-8')

print(f"Plik został pomyślnie zapisany na Dysku Google: {sciezka_dysk_google}")

Plik został pomyślnie zapisany na Dysku Google: /content/drive/MyDrive/Colab Notebooks/obreby_geokodowane_ostateczne.csv


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# 1. Wczytanie pliku z brakami
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/obreby_geokodowane_ostateczne.csv')

# 2. Definicja maski dla pustych rekordów
braki = df['Szerokosc_Y'].isnull()

# 3. Metoda imputacji - wybieramy tylko pierwszą miejscowość kompleksu
def wyczysc_miejscowosc(tekst):
    if pd.isna(tekst):
        return tekst
    # Odcięcie po przecinku
    tekst = str(tekst).split(',')[0].strip()
    # Odcięcie po spójniku ' i '
    tekst = tekst.split(' i ')[0].strip()
    return tekst

# 4. Przygotowanie wektora naprawczego
df.loc[braki, 'Miejscowosc_Clean'] = df.loc[braki, 'Miejscowosc'].apply(wyczysc_miejscowosc)
df.loc[braki, 'Pelny_Adres'] = df.loc[braki, 'Miejscowosc_Clean'] + ', ' + df.loc[braki, 'Wojewodztwo'] + ', ' + df.loc[braki, 'Kraj']

# 5. Ponowna inicjalizacja API
geolocator = Nominatim(user_agent="uniwersytet_korekta_dolnyslask")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

print("Geokodowanie poprawionych rekordów...")
df.loc[braki, 'Lokalizacja'] = df.loc[braki, 'Pelny_Adres'].apply(geocode)
df.loc[braki, 'Szerokosc_Y'] = df.loc[braki, 'Lokalizacja'].apply(lambda loc: loc.latitude if loc else None)
df.loc[braki, 'Dlugosc_X'] = df.loc[braki, 'Lokalizacja'].apply(lambda loc: loc.longitude if loc else None)

# 6. Usunięcie kolumn technicznych
df = df.drop(columns=['Miejscowosc_Clean', 'Pelny_Adres', 'Lokalizacja'], errors='ignore')

# 7. Zapis ostatecznego pliku na Dysku Google
sciezka_finalna = '/content/drive/MyDrive/Colab Notebooks/obreby_final_gotowe.csv'
df.to_csv(sciezka_finalna, index=False, encoding='utf-8')

print(f"Zakończono. Liczba nieprzypisanych współrzędnych: {df['Szerokosc_Y'].isnull().sum()}")
print(f"Plik został zapisany na Dysku Google: {sciezka_finalna}")

Geokodowanie poprawionych rekordów...
Zakończono. Liczba nieprzypisanych współrzędnych: 0
Plik został zapisany na Dysku Google: /content/drive/MyDrive/Colab Notebooks/obreby_final_gotowe.csv
